In [1]:
import numpy as np 
import pandas as pd
import os

In [2]:
import os
import pandas as pd
import re

BASE_PATH = "D:\ISL2\ISL_CSLRT_Corpus"
VIDEOS_PATH = os.path.join(BASE_PATH, "Videos_Sentence_Level")
CSV_PATH = os.path.join(BASE_PATH, "corpus_csv_files")

GLOSS_FILE = os.path.join(CSV_PATH, "ISL Corpus sign glosses.csv")

OUTPUT_META = "dataset_meta.csv"
OUTPUT_SPLIT_DIR = "./dataset_split"

gloss_df = pd.read_csv(GLOSS_FILE)
print(f"✅ Loaded gloss metadata: {len(gloss_df)} entries")
print("Columns in gloss CSV:", gloss_df.columns)

if "Sentence" in gloss_df.columns:
    SENTENCE_COL = "Sentence"
elif "sentence_text" in gloss_df.columns:
    SENTENCE_COL = "sentence_text"
else:
    raise ValueError("Cannot find a column for sentence text in gloss CSV!")

records = [] # BUILD METADATA

# Loop over sentence folders
for sentence_folder in sorted(os.listdir(VIDEOS_PATH)):
    sentence_path = os.path.join(VIDEOS_PATH, sentence_folder)
    if not os.path.isdir(sentence_path):
        continue

    sentence_id = sentence_folder

    # Collect all videos in this sentence folder
    video_files = sorted([f for f in os.listdir(sentence_path) if f.lower().endswith(".mp4")])

    # Assign signer IDs: 7 signers repeating in order
    signer_ids = list(range(1, 8))
    repeat_count = (len(video_files) // len(signer_ids)) + 1
    signer_list = (signer_ids * repeat_count)[:len(video_files)]

    # Loop over videos
    for video_file, signer_id in zip(video_files, signer_list):
        video_path = os.path.join(sentence_path, video_file)

        # Lookup sentence text from gloss CSV
        sentence_text = ""
        if not gloss_df.empty:
            match_text = gloss_df[gloss_df[SENTENCE_COL].astype(str).str.contains(sentence_folder)]
            if not match_text.empty:
                # Pick first match
                sentence_text = match_text.iloc[0][SENTENCE_COL]

        records.append({
            "video_id": video_file,
            "sentence_id": sentence_folder,
            "signer_id": signer_id,
            "sentence_text": sentence_text,
            "sentence_length": None,
            "fps": None,
            "resolution": None,
            "video_path": video_path
        })

# CREATE METADATA DATAFRAME
meta_df = pd.DataFrame(records)
print(f"\n✅ Metadata DataFrame created with {len(meta_df)} records")
print(meta_df.head())

meta_df.to_csv(OUTPUT_META, index=False)
print(f"\n✅ Metadata CSV saved: {OUTPUT_META}")

def assign_split(signer_id):
    if signer_id in [1,2,3,4,5]:
        return "train"
    elif signer_id == 6:
        return "val"
    elif signer_id == 7:
        return "test"
    else:
        return "unknown"

meta_df["split"] = meta_df["signer_id"].apply(assign_split)
print("\n📊 Split distribution:")
print(meta_df["split"].value_counts())

os.makedirs(OUTPUT_SPLIT_DIR, exist_ok=True)

for split in ["train", "val", "test"]:
    split_path = os.path.join(OUTPUT_SPLIT_DIR, split)
    os.makedirs(split_path, exist_ok=True)
    split_df = meta_df[meta_df["split"] == split]
    split_csv = os.path.join(split_path, f"{split}_meta.csv")
    split_df.to_csv(split_csv, index=False)

print(f"\n✅ Split folders and CSVs created in: {OUTPUT_SPLIT_DIR}")
print("   ├── train/train_meta.csv")
print("   ├── val/val_meta.csv")
print("   └── test/test_meta.csv")
print("\n🎯 Phase completed successfully!")



✅ Loaded gloss metadata: 101 entries
Columns in gloss CSV: Index(['Sentence', 'SIGN GLOSSES'], dtype='object')

✅ Metadata DataFrame created with 687 records
                            video_id                sentence_id  signer_id  \
0                       MVI_6503.MP4  He is going into the room          1   
1  he is going into the room (2).mp4  He is going into the room          2   
2      he is going into the room.mp4  He is going into the room          3   
3                       room (2).MP4  He is going into the room          4   
4                       room (3).MP4  He is going into the room          5   

               sentence_text sentence_length   fps resolution  \
0  He is going into the room            None  None       None   
1  He is going into the room            None  None       None   
2  He is going into the room            None  None       None   
3  He is going into the room            None  None       None   
4  He is going into the room            None  No

C:\Users\harsh\AppData\Local\Temp\ipykernel_6964\4000414949.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  match_text = gloss_df[gloss_df[SENTENCE_COL].astype(str).str.contains(sentence_folder)]
C:\Users\harsh\AppData\Local\Temp\ipykernel_6964\4000414949.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  match_text = gloss_df[gloss_df[SENTENCE_COL].astype(str).str.contains(sentence_folder)]
C:\Users\harsh\AppData\Local\Temp\ipykernel_6964\4000414949.py:50: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  match_text = gloss_df[gloss_df[SENTENCE_COL].astype(str).str.contains(sentence_folder)]
C:\Users\harsh\AppData\Local\Temp\ipykernel_6964\4000414949.py:50: UserWarning: This pattern is interpreted as a regular expressio

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
import mediapipe as mp

# ---------------------- CONFIG ----------------------
META_FILE = "dataset_meta.csv"
OUTPUT_DIR = "./preprocessed_data"
TEMPORAL_SMOOTH = True  # Apply moving average filter
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------- STEP 0: LOAD METADATA ----------------------
meta_df = pd.read_csv(META_FILE)
print(f"✅ Loaded metadata with {len(meta_df)} videos")

# Ensure 'split' exists
if 'split' not in meta_df.columns:
    if 'signer_id' not in meta_df.columns:
        raise ValueError("❌ 'signer_id' missing in metadata — rerun Phase 0!")
    def assign_split(signer_id):
        if signer_id in [1, 2, 3, 4, 5]:
            return "train"
        elif signer_id == 6:
            return "val"
        elif signer_id == 7:
            return "test"
        else:
            return "unknown"
    meta_df['split'] = meta_df['signer_id'].apply(assign_split)
    meta_df.to_csv(META_FILE, index=False)
    print("✅ Added 'split' column to metadata")

# Create folders for each split
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(OUTPUT_DIR, split), exist_ok=True)

# ---------------------- STEP 1: INIT MEDIAPIPE ----------------------
mp_holistic = mp.solutions.holistic

# ---------------------- STEP 2: LANDMARK EXTRACTION ----------------------
def extract_landmarks(results):
    """Extract holistic landmarks (pose + hands + face) into a flat array."""
    data = []
    # Pose (33)
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 33 * 3)

    # Left hand (21)
    if results.left_hand_landmarks:
        for lm in results.left_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 21 * 3)

    # Right hand (21)
    if results.right_hand_landmarks:
        for lm in results.right_hand_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 21 * 3)

    # Face (468)
    if results.face_landmarks:
        for lm in results.face_landmarks.landmark:
            data.extend([lm.x, lm.y, lm.z])
    else:
        data.extend([0] * 468 * 3)

    return np.array(data, dtype=np.float32)

# ---------------------- STEP 3: NORMALIZATION ----------------------
def normalize_landmarks(landmarks):
    """Center by torso, scale by shoulder width."""
    landmarks = landmarks.reshape(-1, 3)
    try:
        torso_center = (landmarks[11] + landmarks[12]) / 2
        landmarks -= torso_center
        shoulder_dist = np.linalg.norm(landmarks[11] - landmarks[12])
        if shoulder_dist > 0:
            landmarks /= shoulder_dist
    except Exception:
        pass  # Skip normalization if missing joints
    return landmarks.flatten()

# ---------------------- STEP 4: MAIN PREPROCESSING ----------------------
with mp_holistic.Holistic(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
) as holistic:

    for split in ["train", "val", "test"]:
        split_df = meta_df[meta_df["split"] == split]
        print(f"\n📌 Processing {split} videos: {len(split_df)} videos")

        for _, row in tqdm(split_df.iterrows(), total=len(split_df)):
            video_path = row.get("video_path")
            if not os.path.exists(str(video_path)):
                print(f"⚠️ Skipping missing video: {video_path}")
                continue

            video_id = os.path.splitext(str(row.get("video_id", "unknown")))[0]
            sentence_label = str(row.get("sentence_text", "unknown")).strip()

            cap = cv2.VideoCapture(video_path)
            keypoints_seq = []

            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                results = holistic.process(image_rgb)
                lm = extract_landmarks(results)
                lm = normalize_landmarks(lm)
                keypoints_seq.append(lm)

            cap.release()

            if not keypoints_seq:
                print(f"⚠️ No frames processed for {video_id}, skipping.")
                continue

            keypoints_seq = np.array(keypoints_seq, dtype=np.float32)

            # Temporal smoothing
            if TEMPORAL_SMOOTH and keypoints_seq.shape[0] > 2:
                smoothed = np.copy(keypoints_seq)
                for i in range(1, len(keypoints_seq) - 1):
                    smoothed[i] = (
                        keypoints_seq[i - 1]
                        + keypoints_seq[i]
                        + keypoints_seq[i + 1]
                    ) / 3
                keypoints_seq = smoothed

            # Save keypoints
            npy_path = os.path.join(OUTPUT_DIR, split, f"{video_id}.npy")
            np.save(npy_path, keypoints_seq)

            # Save label (convert to string safely)
            label_path = os.path.join(OUTPUT_DIR, split, f"{video_id}_label.txt")
            with open(label_path, "w", encoding="utf-8") as f:
                f.write(str(sentence_label))

print("\n✅ Phase 1 completed successfully!")
print(f"📂 Preprocessed keypoints & labels saved under: {OUTPUT_DIR}")

✅ Loaded metadata with 687 videos

📌 Processing train videos: 503 videos


100%|██████████| 503/503 [42:28<00:00,  5.07s/it]



📌 Processing val videos: 98 videos


100%|██████████| 98/98 [08:08<00:00,  4.98s/it]



📌 Processing test videos: 86 videos


100%|██████████| 86/86 [04:16<00:00,  2.98s/it]


✅ Phase 1 completed successfully!
📂 Preprocessed keypoints & labels saved under: ./preprocessed_data


In [ ]:
# ---------------------- PHASE 2: MODEL BUILDING ----------------------
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F  # ✅ fixed F undefined
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import Adam
from torch.nn import CTCLoss
import matplotlib.pyplot as plt

# ---------------------- DATASET ----------------------
class SignDataset(Dataset):
    def __init__(self, root_dir):
        self.samples = []
        for file in os.listdir(root_dir):
            if file.endswith(".npy"):
                base = file.replace(".npy", "")
                label_file = os.path.join(root_dir, f"{base}_label.txt")
                if os.path.exists(label_file):
                    with open(label_file, "r", encoding="utf-8") as f:
                        label = f.read().strip().lower()
                    self.samples.append((os.path.join(root_dir, file), label))
        print(f"Loaded {len(self.samples)} samples from {root_dir}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        npy_path, label = self.samples[idx]
        x = np.load(npy_path)                 # (T, F)
        x = torch.tensor(x, dtype=torch.float)
        # Convert characters a-z → 1-26, ignore non-alpha
        y = torch.tensor([ord(c) - 96 for c in label if c.isalpha()], dtype=torch.long)
        return x, y

def collate_fn(batch):
    xs, ys = zip(*batch)
    xs_pad = pad_sequence(xs, batch_first=True)
    ys_pad = pad_sequence(ys, batch_first=True)
    x_lengths = torch.tensor([len(x) for x in xs])
    y_lengths = torch.tensor([len(y) for y in ys])
    return xs_pad, ys_pad, x_lengths, y_lengths

# ---------------------- MODEL ----------------------
class SignToText(nn.Module):
    def __init__(self, input_dim, hidden_dim=256, vocab_size=26):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(input_dim, 256, 5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.3)
        )
        self.lstm = nn.LSTM(256, hidden_dim, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, vocab_size + 1)  # +1 for CTC blank

    def forward(self, x):
        x = x.transpose(1, 2)
        x = self.conv(x)
        x = x.transpose(1, 2)
        out, _ = self.lstm(x)
        out = self.fc(out)
        return out

# ---------------------- TRAINING SETUP ----------------------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

train_ds = SignDataset("./preprocessed_data/train")
val_ds   = SignDataset("./preprocessed_data/val")

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds, batch_size=32, collate_fn=collate_fn)

input_dim = np.load(train_ds.samples[0][0]).shape[1]
model = SignToText(input_dim, hidden_dim=128, vocab_size=26).to(DEVICE)

criterion = CTCLoss(blank=0, zero_infinity=True)
optimizer = Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# num_epochs = 20
train_losses, val_losses = [], []
num_epochs = 60
# (Optionally disable scheduler until val_loss stabilizes)


# ---------------------- TRAINING LOOP ----------------------
for epoch in range(num_epochs):
    # ---------- Training ----------
    model.train()
    epoch_loss = 0.0
    for X, Y, XL, YL in train_loader:
        X, Y = X.to(DEVICE), Y.to(DEVICE)
        optimizer.zero_grad()

        out = model(X)               # (B, T, C)
        out = out.transpose(0, 1)    # (T, B, C) for CTC

        # Adjust input lengths for pooling (MaxPool1d halves time dimension)
        XL_adjusted = XL

        loss = criterion(out, Y, XL_adjusted, YL)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)

    # ---------- Validation ----------
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X, Y, XL, YL in val_loader:
            X, Y = X.to(DEVICE), Y.to(DEVICE)
            out = model(X).transpose(0, 1)
            XL_adjusted = XL
            val_loss += criterion(out, Y, XL_adjusted, YL).item()

    avg_val_loss = val_loss / len(val_loader)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    scheduler.step(avg_val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} → train: {avg_train_loss:.4f}, val: {avg_val_loss:.4f}")

# ---------------------- SAVE MODEL ----------------------
torch.save(model.state_dict(), "model_checkpoint.pth")
print("✅ Model checkpoint saved!")

# ---------------------- PLOT LOSS ----------------------
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.legend()
plt.title("Training Loss Curve")
plt.xlabel("Epoch")
plt.ylabel("CTC Loss")
plt.show()

# ---------------------- SIMPLE INFERENCE ----------------------
def greedy_decode(logits):
    pred = logits.argmax(2).squeeze().cpu().numpy()
    prev = -1
    output = []
    for p in pred:
        if p != prev and p != 0:
            output.append(chr(p + 96))
        prev = p
    return "".join(output)

model.eval()
sample_x, _ = train_ds[0]
with torch.no_grad():
    pred = model(sample_x.unsqueeze(0).to(DEVICE))
print("Predicted text:", greedy_decode(pred))

Loaded 503 samples from ./preprocessed_data/train
Loaded 98 samples from ./preprocessed_data/val
Epoch 1/60 → train: 0.6477, val: -0.0288
Epoch 2/60 → train: 3.1586, val: 3.0969
Epoch 3/60 → train: 3.7464, val: 3.9940
Epoch 4/60 → train: 3.6001, val: 4.3209
Epoch 5/60 → train: 3.3746, val: 4.1728
Epoch 6/60 → train: 3.2877, val: 4.1649
Epoch 7/60 → train: 3.2579, val: 3.9699
Epoch 8/60 → train: 3.2386, val: 4.0519
Epoch 9/60 → train: 3.2269, val: 3.9436
Epoch 10/60 → train: 3.1579, val: 4.1185
Epoch 11/60 → train: 3.1973, val: 4.0109
Epoch 12/60 → train: 3.2172, val: 4.0297
Epoch 13/60 → train: 3.1763, val: 4.0600
Epoch 14/60 → train: 3.1843, val: 4.0003
Epoch 15/60 → train: 3.1633, val: 4.0771
Epoch 16/60 → train: 3.2366, val: 3.9835
Epoch 17/60 → train: 3.1394, val: 4.0631
Epoch 18/60 → train: 3.2297, val: 4.0734
Epoch 19/60 → train: 3.1588, val: 4.0031
Epoch 20/60 → train: 3.1798, val: 4.0548
Epoch 21/60 → train: 3.1835, val: 4.0189
Epoch 22/60 → train: 3.1709, val: 4.0406
Epoch 23/